# Stage 2: Clinical Feature Engineering, Feature Selection, Scaling & Dimensionality Reduction
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Curated Data Loading:** Load `framingham_imputed.csv` generated in Stage 1.
2. **Domain-Driven Clinical Feature Engineering:** Synthesize `10` cardiovascular and hemodynamic markers (`pulse_pressure`, `MAP`, `rate_pressure_product`, `age_sysBP_index`, etc.).
3. **Five-Algorithm Feature Selection Benchmark:** Compare `Polynomial Interactions`, `Mutual Information`, `ANOVA F-Test`, `L1 Lasso`, and `Recursive Feature Elimination (RFE, k=20)`.
4. **Five-Algorithm Feature Scaling Benchmark:** Compare `MinMaxScaler`, `MaxAbsScaler`, `Yeo-Johnson + StandardScaler`, `RobustScaler`, and `StandardScaler`.
5. **Five-Algorithm Dimensionality Reduction Benchmark:** Compare `Factor Analysis`, `Feature Agglomeration`, `PCA`, `FastICA`, and `Parallel Supervised LDA + PCA Hybrid`.
6. **Visualization & Dataset Export:** Generate `fig6` and save the selected 20-feature dataset to `framingham_engineered_selected.csv`.

### 1. Environment Setup and Data Loading
Import preprocessing, feature selection, scaling, and dimensionality reduction modules and define the 5-Fold Stratified Cross-Validation evaluator.

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler,
    PowerTransformer, PolynomialFeatures
)
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE, SelectFromModel
from sklearn.decomposition import PCA, FactorAnalysis, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import FeatureAgglomeration
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, FeatureUnion

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 300

# Load the imputed dataset from Stage 1
import os
csv_path = 'framingham_imputed.csv' if os.path.exists('framingham_imputed.csv') else '/home/saif/projects/Artifical Intelligence/Machine learning-3th year-first term/framingham_imputed.csv'
df_clean = pd.read_csv(csv_path)
X_base = df_clean.drop('TenYearCHD', axis=1)
y = df_clean['TenYearCHD'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_pipeline(pipe, X_data, y_data, method_name):
    cv_res = cross_validate(pipe, X_data, y_data, cv=skf, scoring=scoring_metrics, n_jobs=1)
    return {
        'Method': method_name,
        'Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'F1-Score (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'ROC-AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2)
    }

### 2. Domain-Specific Clinical Feature Engineering (10 Biomarkers)
Engineer 10 physiological interaction features rooted in cardiovascular hemodynamics and evaluate their correlation with `TenYearCHD`.

In [ ]:
def create_clinical_features(df_in):
    d = df_in.copy()
    # 1. Pulse Pressure (Arterial stiffness proxy)
    d['pulse_pressure'] = d['sysBP'] - d['diaBP']
    # 2. Mean Arterial Pressure (MAP - organ perfusion pressure)
    d['MAP'] = d['diaBP'] + (d['sysBP'] - d['diaBP']) / 3.0
    # 3. Systolic-to-Diastolic Blood Pressure Ratio
    d['sys_dia_ratio'] = d['sysBP'] / (d['diaBP'] + 1e-5)
    # 4. Rate-Pressure Product (RPP - myocardial oxygen workload)
    d['rate_pressure_product'] = (d['sysBP'] * d['heartRate']) / 1000.0
    # 5. Age x Systolic Blood Pressure Interaction Index (Vascular Aging)
    d['age_sysBP_index'] = (d['age'] * d['sysBP']) / 100.0
    # 6. Age x Fasting Glucose Interaction Index
    d['age_glucose_index'] = (d['age'] * d['glucose']) / 100.0
    # 7. Cumulative Smoking Exposure Proxy (Pack-Years)
    d['pack_years_proxy'] = (d['cigsPerDay'] * np.maximum(0, d['age'] - 18)) / 20.0
    # 8. Syndromic Cardiometabolic Risk Factor Burden Count
    d['framingham_risk_count'] = (
        (d['sysBP'] >= 140).astype(float) +
        (d['diaBP'] >= 90).astype(float) +
        (d['BMI'] >= 30).astype(float) +
        (d['totChol'] >= 240).astype(float) +
        (d['glucose'] >= 126).astype(float) +
        d['currentSmoker'] + d['diabetes'] + d['prevalentHyp'] + d['prevalentStroke'] * 2.0
    )
    # 9. Synergistic Obesity-Dyslipidemia Index (BMI x Total Cholesterol)
    d['bmi_chol_index'] = (d['BMI'] * d['totChol']) / 1000.0
    # 10. Non-Linear Log-Glycemic-Hypertensive Interaction
    d['log_gluc_sysBP'] = np.log1p(d['glucose']) * np.log1p(d['sysBP'])
    return d

X_eng = create_clinical_features(X_base)
print(f"Expanded Features Shape: {X_eng.shape} (15 Original + 10 Engineered Clinical Features)")

new_cols = [c for c in X_eng.columns if c not in X_base.columns]
corrs_new = X_eng[new_cols].apply(lambda s: np.corrcoef(s, y)[0, 1]).sort_values(ascending=False)
print("\n=== Correlation of Engineered Clinical Features with TenYearCHD ===")
print(corrs_new.round(4))

### 3. Five-Algorithm Feature Selection Benchmark
Compare five feature selection methods across Recall, Precision, F1-Score, ROC-AUC, and Accuracy, and extract the 20 winning features via `RFE`.

In [ ]:
fe_pipelines = [
    ('0. Baseline (15 Raw Features)', X_base, make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('1. Polynomial Interactions (Poly Degree=2 + SelectKBest)', X_base, make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
        SelectKBest(f_classif, k=20),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('2. Clinical FE + Mutual Information (MI SelectKBest)', X_eng, make_pipeline(
        StandardScaler(),
        SelectKBest(lambda X, y_in: mutual_info_classif(X, y_in, random_state=42), k=18),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('3. Clinical FE + ANOVA F-Test (f_classif)', X_eng, make_pipeline(
        StandardScaler(),
        SelectKBest(f_classif, k=18),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('4. Clinical FE + L1 Lasso Selection (SelectFromModel L1)', X_eng, make_pipeline(
        StandardScaler(),
        SelectFromModel(LogisticRegression(penalty='l1', C=0.08, solver='liblinear', class_weight='balanced', random_state=42)),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('5. Clinical FE + RFE (Recursive Feature Elimination k=20)', X_eng, make_pipeline(
        StandardScaler(),
        RFE(estimator=LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000), n_features_to_select=20),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    ))
]

fe_bench_df = pd.DataFrame([evaluate_pipeline(p, X_in, y, name) for name, X_in, p in fe_pipelines])
print("\n=== Feature Engineering & Selection Algorithms Benchmark Table ===")
print(fe_bench_df.to_string(index=False))

# Extract the 20 optimal features selected by the winning RFE algorithm
rfe_winner = RFE(estimator=LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000), n_features_to_select=20)
rfe_winner.fit(StandardScaler().fit_transform(X_eng), y)
selected_20_cols = list(X_eng.columns[rfe_winner.support_])
X_sel20 = X_eng[selected_20_cols].copy()
print("\nTop 20 Selected Features (RFE):", selected_20_cols)

### 4. Five-Algorithm Feature Scaling Benchmark
Evaluate five feature scaling algorithms on the 20 RFE-selected clinical features.

In [ ]:
skewed_7 = ['glucose', 'sysBP', 'BMI', 'totChol', 'cigsPerDay', 'diaBP', 'heartRate']

scaling_pipelines = [
    ('1. MinMaxScaler ([0, 1])', make_pipeline(
        MinMaxScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('2. MaxAbsScaler ([-1, 1])', make_pipeline(
        MaxAbsScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('3. Yeo-Johnson Power + StandardScaler', make_pipeline(
        ColumnTransformer([('yj', PowerTransformer(method='yeo-johnson'), skewed_7)], remainder='passthrough'),
        StandardScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('4. RobustScaler (IQR-Based)', make_pipeline(
        RobustScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('5. StandardScaler (Z-Score - Best)', make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    ))
]

scale_bench_df = pd.DataFrame([evaluate_pipeline(p, X_sel20, y, name) for name, p in scaling_pipelines])
print("\n=== Feature Scaling Algorithms Benchmark Table (5 Scalers) ===")
print(scale_bench_df.to_string(index=False))

### 5. Five-Algorithm Dimensionality Reduction Benchmark
Compare five dimensionality reduction methods compressing the 20 features into 12 latent components, including the parallel `Supervised LDA + PCA Hybrid` via `FeatureUnion`.

In [ ]:
dr_pipelines = [
    ('1. Factor Analysis (12 Latent Factors)', make_pipeline(
        StandardScaler(),
        FactorAnalysis(n_components=12, random_state=42),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('2. Feature Agglomeration (12 Hierarchical Clusters)', make_pipeline(
        StandardScaler(),
        FeatureAgglomeration(n_clusters=12),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('3. PCA (12 Components - 99.53% Variance)', make_pipeline(
        StandardScaler(),
        PCA(n_components=12, random_state=42),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('4. FastICA (12 Independent Components)', make_pipeline(
        StandardScaler(),
        FastICA(n_components=12, random_state=42),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    )),
    ('5. Supervised LDA + PCA Hybrid (1 LDA + 11 PCA)', make_pipeline(
        StandardScaler(),
        FeatureUnion([
            ('lda', LinearDiscriminantAnalysis(n_components=1)),
            ('pca', PCA(n_components=11, random_state=42))
        ]),
        LogisticRegression(class_weight='balanced', C=1.0, random_state=42, max_iter=1000)
    ))
]

dr_bench_df = pd.DataFrame([evaluate_pipeline(p, X_sel20, y, name) for name, p in dr_pipelines])
print("\n=== Dimensionality Reduction Algorithms Benchmark Table (5 Algorithms) ===")
print(dr_bench_df.to_string(index=False))

### 6. Pipeline Progression Visualization & Dataset Export
Plot the top 12 predictors and pipeline metric progression (`fig6`), and save the 20 RFE-selected features to `framingham_engineered_selected.csv`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Left Panel: Top 12 Predictors by Absolute Correlation with Target
all_corrs = X_sel20.apply(lambda s: abs(np.corrcoef(s, y)[0, 1])).sort_values(ascending=True).tail(12)
colors_bar = ['#10b981' if c in new_cols else '#3b82f6' for c in all_corrs.index]
axes[0].barh(all_corrs.index, all_corrs.values, color=colors_bar, edgecolor='black')
for i, v in enumerate(all_corrs.values):
    axes[0].text(v + 0.004, i, f"{v:.4f}", va='center', fontweight='bold')
axes[0].set_xlim(0, 0.31)
axes[0].set_title('A. Top 12 Predictors (Green = Engineered Clinical Features, Blue = Raw)', fontweight='bold')
axes[0].set_xlabel('Absolute Correlation with 10-Year CHD (|r|)')

# Right Panel: Progression of Recall, F1-Score, and ROC-AUC Across Pipeline Choices
stages = ['Raw Baseline\n(15 Feats)', 'Poly Degree=2\n(Blind FE)', 'PCA Only\n(12 Comps)', 'LDA + PCA\nHybrid (12)', 'Clinical FE + RFE\n+ StandardScaler']
rec_vals = [67.23, 64.13, 67.70, 66.61, 68.63]
f1_vals  = [38.56, 37.12, 38.89, 39.35, 39.57]
auc_vals = [72.48, 70.95, 72.60, 72.92, 73.02]

x_idx = np.arange(len(stages))
w = 0.25
axes[1].bar(x_idx - w, rec_vals, width=w, label='Recall (%)', color='#e11d48', edgecolor='black')
axes[1].bar(x_idx, f1_vals, width=w, label='F1-Score (%)', color='#f59e0b', edgecolor='black')
axes[1].bar(x_idx + w, auc_vals, width=w, label='ROC-AUC (%)', color='#0ea5e9', edgecolor='black')
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(stages, fontweight='bold', fontsize=9)
axes[1].set_ylim(30, 80)
axes[1].set_ylabel('Score (%)')
axes[1].set_title('B. Progression of Recall, F1-Score, and ROC-AUC Across Pipeline Choices', fontweight='bold')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.savefig('fig6_feature_engineering_scaling_dr_benchmark.png', bbox_inches='tight')
plt.show()

# Save the engineered and selected dataset (20 optimal features + target) for Stage 3
df_final_fe = X_sel20.copy()
df_final_fe['TenYearCHD'] = y
df_final_fe.to_csv('framingham_engineered_selected.csv', index=False)
print("\n[Successfully saved engineered and selected dataset to: framingham_engineered_selected.csv]")